In [ ]:
import numpy as np
np.random.seed(42)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import pandas as pd
import time

from processing.load_datasets import load_datasets
from processing.configs import SINGLE_COLOR, COLORS_4

from sklearn.manifold import TSNE
from umap import UMAP
from trimap import TRIMAP
from pacmap import PaCMAP

In [ ]:
# Load datasets
datasets_names = ["rts", "pdl", "ioc", "mjf"]
datasets = load_datasets(datasets_names)

In [ ]:
embeddings_dict = {}
N_SAMPLE = 10000

EMBEDDINGS_FOLDER = "embeddings/params/"

## tSNE - Study of parameters

In [ ]:
perplexities = [5, 10, 50, 100]

for dataset in datasets_names:
    X = datasets[dataset]
    if N_SAMPLE > 0 and N_SAMPLE < X.shape[0]:
        rng = np.random.default_rng(42)
        X = X[rng.choice(X.shape[0], N_SAMPLE, replace=False)]
        
    embeddings = []
    for perplexity in perplexities:
        print(f"Running t-SNE with perplexity={perplexity}...")
        embeddings.append(TSNE(perplexity=perplexity).fit_transform(X))
        
    embeddings_dict[f"{dataset}_tsne"] = embeddings

In [ ]:
for dataset in datasets_names:
    key = f"{dataset}_tsne"
    fig, ax = plt.subplots(1, 4, figsize=(20, 5))
    for i, perplexity in enumerate(perplexities):
        ax[i].scatter(embeddings_dict[key][i][:, 0], embeddings_dict[key][i][:, 1], s=0.1, color=SINGLE_COLOR)
        ax[i].set_title(f'Perplexity={perplexity}', fontsize=16, fontweight='bold')
        ax[i].set_xticks([])
        ax[i].set_yticks([])
    plt.tight_layout()
    plt.savefig(f"images/params/tsne_params_{dataset}.png", dpi=300, bbox_inches='tight')
    plt.show()

## UMAP - Study of parameters

In [ ]:
n_neighbors = [5, 30, 50, 100]
min_distances = [0.1, 0.5, 0.99]

for dataset in datasets_names:
    X = datasets[dataset]
    if N_SAMPLE > 0 and N_SAMPLE < X.shape[0]:
        rng = np.random.default_rng(42)
        X = X[rng.choice(X.shape[0], N_SAMPLE, replace=False)]

    embeddings = []
    for d in min_distances:
        for n in n_neighbors:
            print(f"Running UMAP with n_neighbors={n} and min_dist={d}...")
            embeddings.append(UMAP(n_neighbors=n, min_dist=d).fit_transform(X))
            
    embeddings_dict[f"{dataset}_umap"] = embeddings

In [ ]:
for dataset in datasets_names:
    key = f"{dataset}_umap"
    fig, axs = plt.subplots(len(min_distances), len(n_neighbors), figsize=(20, 15))
    counter = 0
    for i, min_dist in enumerate(min_distances):
        for j, n in enumerate(n_neighbors):
            axs[i,j].scatter(embeddings_dict[key][counter][:, 0], embeddings_dict[key][counter][:, 1], s=0.1, color=SINGLE_COLOR)
            #axs[i,j].set_title(f'n_neighbors={n}, min_dist={min_dist}', fontsize=16, fontweight='bold')
            axs[i,j].set_xticks([])
            axs[i,j].set_yticks([])
            counter += 1
            
    for i, min_dist in enumerate(min_distances):
        axs[i,0].set_ylabel(f'min_dist={min_dist}', fontsize=16, fontweight='bold')
    for j, n in enumerate(n_neighbors):
        axs[0,j].set_title(f'n_neighbors={n}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"images/params/umap_params_{dataset}.png", dpi=300, bbox_inches='tight')
    plt.show()

## TRIMAP - Study of parameters

In [ ]:
c_values = [5, 30, 50, 100]

for dataset in datasets_names:
    X = datasets[dataset]
    if N_SAMPLE > 0 and N_SAMPLE < X.shape[0]:
        rng = np.random.default_rng(42)
        X = X[rng.choice(X.shape[0], N_SAMPLE, replace=False)]

    embeddings = []
    for c in c_values:
        print(f"Running TRIMAP with n_inliers={2*c}, n_outliers={c}, n_random={c}...")
        X_low = TRIMAP(n_inliers=2*c, n_outliers=c, n_random=c).fit_transform(X)

        save_path = os.path.join(EMBEDDINGS_FOLDER, f"{dataset}_trimap_c{c}.npy")
        if not os.path.exists(EMBEDDINGS_FOLDER):
            os.makedirs(EMBEDDINGS_FOLDER)
        np.save(save_path, X_low)

        embeddings.append(X_low)
        
    embeddings_dict[f"{dataset}_trimap"] = embeddings

In [ ]:
for dataset in datasets_names:
    key = f"{dataset}_trimap"
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    for i, c in enumerate(c_values):
        axs[i].scatter(embeddings_dict[key][i][:, 0], embeddings_dict[key][i][:, 1], s=0.1, color=SINGLE_COLOR)
        axs[i].set_title(f'triplets={c}(2,1,1)', fontsize=16, fontweight='bold')
        axs[i].set_xticks([])
        axs[i].set_yticks([])
    plt.tight_layout()
    plt.savefig(f"images/params/trimap_params_{dataset}.png", dpi=300, bbox_inches='tight')
    plt.show()

## PACMAP - Study of parameters

In [ ]:
mn_ratios = [0.1, 0.5, 1, 5]
fp_ratios = [0.5, 2, 5, 10]

for dataset in datasets_names:
    X = datasets[dataset]
    if N_SAMPLE > 0 and N_SAMPLE < X.shape[0]:
        rng = np.random.default_rng(42)
        X = X[rng.choice(X.shape[0], N_SAMPLE, replace=False)]

    embeddings = []
    for mn in mn_ratios:
        for fp in fp_ratios:
            print(f"Running PaCMAP with MN_ratio={mn} and FP_ratio={fp}...")
            embeddings.append(PaCMAP(MN_ratio=mn, FP_ratio=fp).fit_transform(X))
            
    embeddings_dict[f"{dataset}_pacmap_ratio"] = embeddings

In [ ]:
for dataset in datasets_names:
    key = f"{dataset}_pacmap_ratio"
    fig, axs = plt.subplots(len(mn_ratios), len(fp_ratios), figsize=(20, 20))
    counter = 0
    for i, mn in enumerate(mn_ratios):
        for j, fp in enumerate(fp_ratios):
            axs[i,j].scatter(embeddings_dict[key][counter][:, 0], embeddings_dict[key][counter][:, 1], s=0.1, color=SINGLE_COLOR)
            axs[i,j].set_xticks([])
            axs[i,j].set_yticks([])
            counter += 1
            
    for i, mn in enumerate(mn_ratios):
        axs[i,0].set_ylabel(f'MN_ratio={mn}', fontsize=16, fontweight='bold')
    for j, fp in enumerate(fp_ratios):
        axs[0,j].set_title(f'FP_ratio={fp}', fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f"images/params/pacmap_params_ratios_{dataset}.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
n_values = [5, 30, 50, 100]

for dataset in datasets_names:
    X = datasets[dataset]
    if N_SAMPLE > 0 and N_SAMPLE < X.shape[0]:
        rng = np.random.default_rng(42)
        X = X[rng.choice(X.shape[0], N_SAMPLE, replace=False)]

    embeddings = []
    for n in n_values:
        print(f"Running PaCMAP with n_neighbors={n}...")
        embeddings.append(PaCMAP(n_neighbors=n).fit_transform(X))
        
    embeddings_dict[f"{dataset}_pacmap_n"] = embeddings

In [ ]:
for dataset in datasets_names:
    key = f"{dataset}_pacmap_n"
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    for i, n in enumerate(n_values):
        axs[i].scatter(embeddings_dict[key][i][:, 0], embeddings_dict[key][i][:, 1], s=0.1, color=SINGLE_COLOR)
        axs[i].set_title(f'n_neighbors={n}', fontsize=16, fontweight='bold')
        axs[i].set_xticks([])
        axs[i].set_yticks([])

    plt.tight_layout()
    plt.savefig(f"images/params/pacmap_params_n_{dataset}.png", dpi=300, bbox_inches='tight')
    plt.show()

# Parameters stability evaluation

In [ ]:
# Compute Procustes distance between embeddings
from scipy.spatial import procrustes

def procrustes_distance(X, Y):
    _, _, dist = procrustes(X, Y)
    return dist

# Compute pairwise Procrustes distances between embeddings
def compute_procrustes_distances(embeddings):
    n = len(embeddings)
    distances = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            distances[i, j] = procrustes_distance(embeddings[i], embeddings[j])
    return distances

In [ ]:
embeddings_procustes_d = {key:compute_procrustes_distances(embeddings) for key, embeddings in embeddings_dict.items()}
embeddings_procustes_d_eval = {key:(np.mean(distances), np.std(distances)) for key, distances in embeddings_procustes_d.items()}

In [ ]:
embeddings_procustes_d_eval = pd.DataFrame(embeddings_procustes_d_eval, index=["mean", "std"]).T.reset_index().rename(columns={"index":"dataset_algo"})
embeddings_procustes_d_eval["dataset"] = embeddings_procustes_d_eval["dataset_algo"].apply(lambda x: x.split("_")[0])
embeddings_procustes_d_eval["algo"] = embeddings_procustes_d_eval["dataset_algo"].apply(lambda x: x.split("_")[1])

In [ ]:
barwidth = 0.2
algo_names = ["tsne", "umap", "trimap", "pacmap"]
n_datasets = len(datasets_names)
n_algos = len(algo_names)

legend_patches = [mpatches.Patch(color=COLORS_4(i), label=algo.upper()) for i, algo in enumerate(algo_names)]

# Plot neighborhood preservation on single plot
fig, ax = plt.subplots(figsize=(10, 4))
for i, dataset in enumerate(datasets_names):
    for j, algo in enumerate(algo_names):
        df = embeddings_procustes_d_eval[(embeddings_procustes_d_eval["dataset"] == dataset) & (embeddings_procustes_d_eval["algo"] == algo)]
        ax.bar(i + j * barwidth, df["mean"], yerr=df["std"], width=barwidth, label=f"{algo}", color = COLORS_4(j))
ax.set_xticks(np.arange(n_datasets) + barwidth * (n_algos - 1) / 2)
ax.set_xticklabels([w.upper() for w in datasets_names], fontsize = 12, fontweight='bold')
ax.set_ylabel("Procustes distance", fontsize = 12, fontweight='bold')
ax.legend(handles=legend_patches, loc="upper left", bbox_to_anchor=(1, 1))
plt.show()